# 3 &mdash; Ausgrid: rooftop PV generation from real measurements

Demonstration of **Section 4.3** of [*B-LSTM-MIONet*](https://arxiv.org/abs/2311.16519),
arXiv:2311.16519 &mdash; the power-engineering application.

## The dataset

Ausgrid, an Australian distribution network operator, published *Solar home electricity data*:
half-hourly load and rooftop-PV measurements for **300 de-identified customers** over 36 months,
1 July 2010 to 30 June 2013, in three yearly CSV files. The paper uses only the **gross
generation** category (`GG`), the power produced by the PV panels, because unlike household load
it is driven by weather and geography rather than by habits.

Each CSV row is one customer on one day: 48 half-hour readings in kWh. The paper keeps the
daylight columns (positions 18 to 38, the readings from 07:00 to 17:00), interpolates them onto a grid of
spacing $h = 0.05$ h giving 210 points per day, and treats each daily profile as one trajectory of
an **autonomous** system &mdash; the LSTM branch is fed the history of the generation itself.
Customers 1-50 of the first year make the training set (366 &times; 50 &times; 5 = 91500 quartets);
customers 51-60 and 61-70 are the two test groups.

## Getting the data

The CSV files are **Ausgrid's and are not part of this repository**. Ausgrid no longer hosts
the download page (the dataset is described in [Ratnam et al., 2017](https://doi.org/10.1080/14786451.2015.1100196)); the three yearly
files are in the authors' archive, see `scripts/download_data.sh`. Unzip it and point
`data.ausgrid.csv_paths` in `configs/ausgrid.yaml` at
the extracted files. The paths shipped in the config are

```
data/Ausgrid/Solar home half-hour data - 1 July 2010 to 30 June 2011/2010-2011 Solar home electricity data.csv
data/Ausgrid/Solar home half-hour data - 1 July 2011 to 30 June 2012/2011-2012 Solar home electricity data v2.csv
data/Ausgrid/Solar home half-hour data - 1 July 2012 to 30 June 2013/2012-2013 Solar home electricity data v2.csv
```

**If those files are missing, this notebook still runs**: it writes a small *synthetic* CSV with
the same layout into `data/` and says so loudly. The synthetic profiles are smooth bell curves, so
the errors below say nothing about the real problem &mdash; they only prove the pipeline works.

In [ ]:
import datetime
import logging
import os
import warnings
from pathlib import Path

# Locate the repository root, whether the kernel started in notebooks/ or at the root.
ROOT = Path.cwd().resolve()
while not (ROOT / "configs" / "ausgrid.yaml").is_file():
    if ROOT == ROOT.parent:
        raise RuntimeError("could not find the repository root (no configs/ausgrid.yaml above cwd)")
    ROOT = ROOT.parent

# Paths inside a configuration file are relative to the working directory, so work from the
# repository root: data/, mlruns/ and figures/ are then created there (all three are git-ignored).
os.chdir(ROOT)
os.environ["MLFLOW_TRACKING_URI"] = str(ROOT / "mlruns")
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"  # set before mlflow is imported
warnings.filterwarnings("ignore")
logging.getLogger("mlflow").setLevel(logging.ERROR)

print("repository root     :", ROOT)
print("MLFLOW_TRACKING_URI :", os.environ["MLFLOW_TRACKING_URI"])

In [ ]:
import copy

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from blstm_mionet.config import load_config
from blstm_mionet.data.ausgrid import select_ausgrid_data
from blstm_mionet.data.datasets import prepare_torch_dataset, split_dataset
from blstm_mionet.data.generate import load_dataset, save_dataset
from blstm_mionet.evaluation.evaluate import evaluate_single_step
from blstm_mionet.models import build_model, dataset_preparer_for
from blstm_mionet.training import tracking
from blstm_mionet.training.trainer import train_adam
from blstm_mionet.utils.device import resolve_device
from blstm_mionet.utils.seed import set_seed

# blstm_mionet.evaluation.plotting selects the non-interactive "Agg" backend when it is
# imported, so ask for the inline backend again afterwards.
%matplotlib inline

print("torch               :", torch.__version__)

## Settings: `QUICK` versus the paper

| | `QUICK` | paper, Sec. 4.3 (`configs/ausgrid.yaml`) |
| --- | --- | --- |
| training customers | 1-4 | 1-50 |
| training days | 30 (July 2010) | 366 (1 Jul 2010 - 30 Jun 2011) |
| test customers | 5-6, same days | 51-60 and 61-70, same year |
| daylight columns kept | 18:39 (21 half-hour readings) | 18:39 |
| interpolation | `delta_t_idxs = 0.1` &rarr; 210 points, $h = 0.05$ h | same |
| masks per daily profile | 20 | 5, giving $N_{\text{train}} = 91500$ |
| max spacing $h_{\max}$ | 0.5 h (`search_len = 10`) | 0.5 h |
| branch / trunk | width 100, depth 3 | width 200, depth 3 |
| LSTM | 64 units, 1 layer | 100 units, 2 layers |
| epochs | 5 | 1000, early stopping after 80 |

## The CSV layout the loader expects

`select_ausgrid_data` calls `pd.read_csv(path, header=1)`, so line 1 of the file is a **title
line** and line 2 is the header. The header must contain `Customer`, `Consumption Category` and
`date` (Australian day-first `1/07/2010`), followed by the 48 half-hour reading columns, so that
`iloc[:, 18:39]` lands on the daylight window:

```
Solar home half-hour data ...                                <- title line, skipped
Customer,Generator Capacity,Postcode,Consumption Category,date,0:30,1:00,...,23:30,0:00
1,1.54,2076,GG,1/07/2010,0,0,...,0,0
1,1.54,2076,GC,1/07/2010,0.114,0.099,...,0.2,0.18
```

The next cell checks whether the configured files exist and, if they do not, writes a synthetic
file with exactly that layout.

In [ ]:
QUICK = True

QUICK_OVERRIDES = [
    "data.ausgrid.customer_id=[1, 2, 3, 4]",
    'data.ausgrid.start_date="2010-07-01"',
    'data.ausgrid.end_date="2010-07-30"',
    "data.output=data/ausgrid_demo_train.npy",
    # a smaller operator
    "model.branch_state.width=100",
    "model.branch_memory.width=100",
    "model.branch_memory.lstm_size=64",
    "model.branch_memory.lstm_layer_num=1",
    "model.trunk.width=100",
    # training
    "training.datafile=data/ausgrid_demo_train.npy",
    "training.search_num=20",
    "training.batch_size=64",
    "training.epochs=5",
    "training.device=cpu",
    "training.run_name=ausgrid_quick",
    "training.registered_model_name=null",
    "training.plot_trajs=false",
    # inference
    "inference.search_num=20",
    "inference.device=cpu",
    "inference.plot_trajs=false",
]

# Paper scale: configs/ausgrid.yaml already carries the Section 4.3 values.
FULL_OVERRIDES = [
    "training.device=0",
    "inference.device=0",
]

config = load_config(
    ROOT / "configs" / "ausgrid.yaml", QUICK_OVERRIDES if QUICK else FULL_OVERRIDES
)

TRAIN_FILE = config.data.output
TEST_FILE = "data/ausgrid_demo_test.npy" if QUICK else "data/ausgrid_cust_51-60.npy"

In [ ]:
SYNTHETIC_CSV = Path("data/ausgrid_synthetic_half_hour.csv")


def write_synthetic_ausgrid_csv(path, n_customers=6, start="2010-07-01", n_days=30, seed=7):
    # Write a CSV with the Ausgrid layout: title line, header, 48 half-hour columns.
    rng = np.random.default_rng(seed)
    labels = [f"{(k * 30 // 60) % 24}:{k * 30 % 60:02d}" for k in range(1, 49)]
    hours = np.arange(1, 49) * 0.5
    daylight = np.exp(-(((hours - 12.5) / 3.2) ** 2))
    daylight[(hours < 6.5) | (hours > 18.5)] = 0.0

    first_day = datetime.date.fromisoformat(start)
    rows = []
    for customer in range(1, n_customers + 1):
        capacity = round(float(rng.uniform(1.0, 4.0)), 2)
        for day in range(n_days):
            date = first_day + datetime.timedelta(days=day)
            cloud = float(rng.uniform(0.55, 1.0))
            profile = 0.18 * capacity * cloud * daylight
            profile = np.clip(profile * (1 + 0.04 * rng.standard_normal(48)), 0.0, None)
            rows.append([
                customer, capacity, 2000 + customer, "GG",
                f"{date.day}/{date.month:02d}/{date.year}", *np.round(profile, 4),
            ])

    frame = pd.DataFrame(
        rows,
        columns=["Customer", "Generator Capacity", "Postcode", "Consumption Category", "date", *labels],
    )
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="") as handle:
        handle.write(
            "SYNTHETIC half-hour solar home data written by notebooks/03_ausgrid.ipynb "
            "- NOT the Ausgrid release\n"
        )
        frame.to_csv(handle, index=False)
    return frame


missing = [p for p in config.data.ausgrid.csv_paths if not Path(p).is_file()]
USING_SYNTHETIC = bool(missing)

if USING_SYNTHETIC:
    print("=" * 78)
    print("!! The licensed Ausgrid CSV files listed in configs/ausgrid.yaml were NOT found:")
    for path in missing:
        print(f"     missing: {path}")
    print("!! Falling back to SYNTHETIC data with the same layout.  Every number produced")
    print("!! below is a pipeline smoke test and says nothing about the real problem.")
    print("!! See the first markdown cell for where to download the real files.")
    print("=" * 78)
    write_synthetic_ausgrid_csv(SYNTHETIC_CSV)
    config.data.ausgrid.csv_paths = [str(SYNTHETIC_CSV)]
    print(f"\nwrote {SYNTHETIC_CSV}")
else:
    print("Using the licensed Ausgrid CSV files listed in configs/ausgrid.yaml:")
    for path in config.data.ausgrid.csv_paths:
        print(f"   {path}")

first_csv = Path(config.data.ausgrid.csv_paths[0])
with first_csv.open() as handle:
    for line_no, line in enumerate(handle):
        if line_no > 2:
            break
        print(f"\nline {line_no}: {line.strip()[:160]} ...")

## 1. Select, filter and resample the daily profiles

`select_ausgrid_data` concatenates the CSV files, keeps the rows matching the customer ids, the
date window and the `GG` category, slices `iloc[:, column_start:column_end]` (the daylight
columns), drops daily records that are more than 20 % zeros, and finally interpolates every
remaining record with a cubic spline onto a grid `delta_t_idxs` times finer. The result has the
same shape convention as the ODE datasets: `x` of shape `[N_time, 1, N_traj]` and `t` in hours.

```bash
blstm-mionet generate --config configs/ausgrid.yaml \
    --set 'data.ausgrid.csv_paths=[data/ausgrid_synthetic_half_hour.csv]' \
    --set 'data.ausgrid.customer_id=[1, 2, 3, 4]' \
    --set 'data.ausgrid.start_date="2010-07-01"' \
    --set 'data.ausgrid.end_date="2010-07-30"' \
    --output data/ausgrid_demo_train.npy
```

In [ ]:
def make_dataset(base, customers, output):
    # Select the daily profiles of ``customers`` and save them to ``output``.
    cfg = copy.deepcopy(base)
    cfg.ausgrid.customer_id = customers
    cfg.output = output
    set_seed(cfg.seed)
    data = select_ausgrid_data(
        csv_paths=cfg.ausgrid.csv_paths,
        cust_id=cfg.ausgrid.customer_id,
        start_date=cfg.ausgrid.start_date,
        end_date=cfg.ausgrid.end_date,
        category=cfg.ausgrid.category,
        delta_t_idxs=cfg.ausgrid.delta_t_idxs,
        column_start=cfg.ausgrid.column_start,
        column_end=cfg.ausgrid.column_end,
        min_nonzero_fraction=cfg.ausgrid.min_nonzero_fraction,
        sample_interval_hours=cfg.ausgrid.sample_interval_hours,
    )
    save_dataset(data, cfg.output)
    print(f"{output}: x {data['x'].shape}, t {data['t'].shape}\n")
    return data


TEST_CUSTOMERS = [5, 6] if QUICK else list(range(51, 61))

train_raw = make_dataset(config.data, config.data.ausgrid.customer_id, TRAIN_FILE)
test_raw = make_dataset(config.data, TEST_CUSTOMERS, TEST_FILE)

In [ ]:
hours = np.asarray(train_raw["t"]).squeeze()
profiles = np.asarray(train_raw["x"])[:, 0, :]

fig, ax = plt.subplots(figsize=(8.5, 4))
for idx in range(0, min(profiles.shape[1], 60), 12):
    ax.plot(hours, profiles[:, idx], lw=1.2, label=f"daily profile {idx}")
ax.set_xlabel("$t$ (hours after the first daylight column)")
ax.set_ylabel("$x(t)$ (kWh per half hour)")
ax.set_title(("SYNTHETIC" if USING_SYNTHETIC else "Ausgrid") + " gross PV generation, resampled at h = 0.05 h")
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

print(f"{profiles.shape[1]} daily profiles x {profiles.shape[0]} interpolated points, "
      f"t from {hours[0]:.2f} h to {hours[-1]:.2f} h")

## 2. A look at the masking (paper Sec. 3.4)

This system is treated as autonomous, so the LSTM branch is fed the masked history of the
generation itself, $x_{(t_0,t_n]}$ (paper Eq. 5). The only difference from the Lorenz notebook is
the time scale: `search_len = 10` interpolated steps is $h_{\max} = 0.5$ hours, and the cut $t_n$
is drawn uniformly from the first `N_time - 2 * search_len` grid points, i.e. from the start of the
daylight window up to about half an hour before its end.

In [ ]:
preparer = dataset_preparer_for(config.model.architecture)
print(f"{config.model.architecture} is prepared by {preparer.__name__}()\n")

train_split, _ = split_dataset(train_raw, test_size=0.0)
input_masked, x_n, x_next, t_params = preparer(
    data=train_split,
    state_component=config.training.state_component,
    search_len=config.training.search_len,
    search_num=config.training.search_num,
    search_random=config.training.search_random,
    offset=config.training.offset,
    t_max=config.training.t_max,
)

n_traj = train_split[1].shape[0]
t_grid = np.asarray(train_split[2]).squeeze()
dt = float(t_grid[1] - t_grid[0])

SAMPLE = 0
picks = [s * n_traj + SAMPLE for s in range(3)]
colours = ["tab:blue", "tab:orange", "tab:green"]

fig, ax = plt.subplots(figsize=(9.5, 4.2))
ax.plot(t_grid, train_split[1][SAMPLE, :, 0], color="0.8", lw=1.0, label="full daily profile")
for colour, k in zip(colours, picks, strict=True):
    t_n = t_params[k, 0]
    n_hist = int(round(t_n / dt)) + 1
    ax.plot(t_grid[:n_hist], input_masked[k, :n_hist, 0], color=colour, lw=1.6,
            label=f"masked history, $t_n$ = {t_n:.2f} h")
    ax.plot(t_n, x_n[k, 0], "o", color=colour, ms=7)
    ax.plot(t_params[k, 2], x_next[k, 0], "*", color=colour, ms=13)
ax.set_xlabel("$t$ (hours)")
ax.set_ylabel("$x(t)$ (kWh)")
ax.set_title("Three masks of the same daily profile (paper Sec. 3.4)\n"
             "circle: current state $x(t_n)$   star: target $x(t_n+h)$")
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

print(f"{input_masked.shape[0]} quartets = {config.training.search_num} masks x {n_traj} daily profiles")
print(f"h_max = {config.training.search_len * dt:.2f} hours")

## 3. Build and train the operator

```bash
blstm-mionet train --config configs/ausgrid.yaml \
    --data data/ausgrid_demo_train.npy --epochs 5 --device cpu --no-plot
```

In [ ]:
set_seed()
device = resolve_device(config.training.device, verbose=True)

train_only, _ = split_dataset(train_raw, test_size=config.training.holdout_size)
train_dataset, _, state_feature_num = prepare_torch_dataset(
    train_only,
    preparer,
    state_component=config.training.state_component,
    search_len=config.training.search_len,
    search_num=config.training.search_num,
    search_random=config.training.search_random,
    offset=config.training.offset,
    t_max=config.training.t_max,
    scale_mode=config.training.scale_mode,
    device=device,
)

model = build_model(config.model, state_feature_num)
print(model)
print(f"\n{sum(p.numel() for p in model.parameters()):,} trainable parameters")

In [ ]:
tracking.configure_tracking(config.tracking.uri)
with tracking.start_run(config.training.experiment_name, config.training.run_name) as run:
    RUN_ID = run.info.run_id
    tracking.log_config(config)
    history = train_adam(config.training, model, train_dataset, device)

MODEL_URI = history["model_uri"]
print(f"\nbest val_loss  : {history['best_metric']['val_loss']:.6f} (epoch {history['best_epoch']})")
print(f"MLflow run uri : runs:/{RUN_ID}")
print(f"model uri      : {MODEL_URI}")

## 4. Evaluate on the held-out customers

Table 5 of the paper reports 1.23 % mean $L_2$ for customers 51-60 and 1.33 % for customers 61-70,
each over a full year of daily profiles.

```bash
blstm-mionet infer --config configs/ausgrid.yaml \
    --data data/ausgrid_demo_test.npy --model runs:/<run id>/model \
    --set inference.search_num=20 --device cpu
```

In [ ]:
inference = config.inference
inference.datafile = TEST_FILE

_, test_split = split_dataset(load_dataset(TEST_FILE), test_size=1.0)
test_dataset, _, _ = prepare_torch_dataset(
    test_split,
    preparer,
    state_component=inference.state_component,
    search_len=inference.search_len,
    search_num=inference.search_num,
    search_random=inference.search_random,
    offset=inference.offset,
    t_max=inference.t_max,
    scale_mode=inference.scale_mode,
    device=device,
)

reloaded = tracking.load_model(MODEL_URI, device)
result = evaluate_single_step(inference, reloaded, test_dataset)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, idx in zip(axes, [0, 1], strict=True):
    order = np.argsort(result["t_next"][idx])
    ax.plot(result["t_next"][idx][order], result["y_true"][idx][order],
            color="red", lw=1.4, label="True")
    ax.plot(result["t_next"][idx][order], result["y_pred"][idx][order],
            color="blue", lw=1.4, ls="--", label="LSTM-MIONet")
    ax.set_xlabel("$t$ (hours)")
    ax.set_ylabel("$x(t)$ (kWh)")
    ax.set_title(f"test profile {idx}: $L_2$ = {result['L2_error_mat'][idx] * 100:.2f} %")
    ax.legend(fontsize=9)
fig.tight_layout()
plt.show()

if USING_SYNTHETIC:
    print("Reminder: these curves are SYNTHETIC bell curves, not Ausgrid measurements.")

## Reproducing the paper

With the licensed CSV files in place, `configs/ausgrid.yaml` holds the Section 4.3 setting
unchanged:

```bash
# customers 1-50, 1 Jul 2010 - 30 Jun 2011  ->  N_train = 91500 quartets
blstm-mionet generate --config configs/ausgrid.yaml

# the two test groups
blstm-mionet generate --config configs/ausgrid.yaml \
    --set 'data.ausgrid.customer_id=[51, 52, 53, 54, 55, 56, 57, 58, 59, 60]' \
    --output data/ausgrid_cust_51-60.npy
blstm-mionet generate --config configs/ausgrid.yaml \
    --set 'data.ausgrid.customer_id=[61, 62, 63, 64, 65, 66, 67, 68, 69, 70]' \
    --output data/ausgrid_cust_61-70.npy

# 1000 epochs of Adam with early stopping; registers models:/Ausgrid/latest
blstm-mionet train --config configs/ausgrid.yaml --device 0

# Table 5
blstm-mionet infer --config configs/ausgrid.yaml --model models:/Ausgrid/latest \
    --data data/ausgrid_cust_51-60.npy
blstm-mionet infer --config configs/ausgrid.yaml --model models:/Ausgrid/latest \
    --data data/ausgrid_cust_61-70.npy
```

Published results (Table 5), state $x(t)$ in kWh:

| | customers 51-60 | customers 61-70 |
| --- | --- | --- |
| mean $L_2$ | 1.23 % | 1.33 % |
| st. dev. $L_2$ | 0.67 % | 0.73 % |

Figure 8 of the paper (`assets/paper/fig8_ausgrid_trajectories.png`) shows the corresponding
trajectories.

All runs are recorded under `mlruns/`; browse them with

```bash
MLFLOW_ALLOW_FILE_STORE=true uv run mlflow ui --backend-store-uri mlruns
```

and open <http://127.0.0.1:5000>.